# 11.12 · Frontier briefings (dated)

**Module 11 · Deep Agents** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 1–1.5 h (each briefing 10–15 min) |
| 📋 Prerequisites | The core lesson each briefing extends (listed per briefing). |
| 🔑 Keys | none: the two demos are scripted; the rest is reading |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Recognise** six fast-moving Deep Agents features and what problem each solves.
2. **Try** the two that run offline: forked subagents and `RubricMiddleware`.
3. **Judge** a beta feature: status, minimum version, risk, and whether to put it on a critical path (default: no).

> 🧪 **How to read this lesson.** Every briefing has a **dated status line**. These features change between minor releases: re-check the linked page before you rely on anything here, and log what changed in the research log. Take each briefing right after the core lesson it extends.

## 📚 Official reading (verified 2026-10-16)

| Briefing | Page | After |
|---|---|---|
| F1 Forked subagents | [Subagents · Forked subagents](https://docs.langchain.com/oss/python/deepagents/subagents) | 11.05 |
| F2 Async subagents | [Async subagents](https://docs.langchain.com/oss/python/deepagents/async-subagents) | 11.05 / 11.10 |
| F3 Interpreters and dynamic subagents | [Interpreters](https://docs.langchain.com/oss/python/deepagents/interpreters), [Dynamic subagents](https://docs.langchain.com/oss/python/deepagents/dynamic-subagents) | 11.09 |
| F4 Grading rubrics | [Grading rubrics](https://docs.langchain.com/oss/python/deepagents/rubric) | 11.11 |
| F5 AG-UI, ACP, A2A | [AG-UI](https://docs.langchain.com/oss/python/deepagents/ag-ui), [ACP](https://docs.langchain.com/oss/python/deepagents/acp), [A2A endpoint](https://docs.langchain.com/langsmith/server-a2a) | 11.10 |
| F6 Managed Deep Agents, Deep Agents Code | [Managed Deep Agents](https://docs.langchain.com/langsmith/python/managed-deep-agents-overview), [Deep Agents Code quickstart](https://docs.langchain.com/oss/deepagents/code/quickstart) | 11.11 |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "deepagents>=0.7.23,<0.8"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("deepagents", "langchain",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/11-deep-agents/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GOOGLE_API_KEY', 'GROQ_API_KEY', 'LANGSMITH_API_KEY')
LANGSMITH_PROJECT = 'm11-frontier'
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.deep import deep_key_ready, deep_model_spec, fallback_specs

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_LANGSMITH_KEY = is_set("LANGSMITH_API_KEY")

DEEP_MODEL = deep_model_spec("main")              # MASTERY_DEEP_MODEL (course default: Gemini free tier)
SUBAGENT_MODEL = deep_model_spec("subagent")      # MASTERY_DEEP_SUBAGENT_MODEL
HAVE_DEEP_KEY = deep_key_ready(DEEP_MODEL)        # the key for THAT provider (GOOGLE_API_KEY for google_genai:)
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m11-<lesson>: one tracing project per lesson

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("deep model:", DEEP_MODEL, "| subagents:", SUBAGENT_MODEL)
print("fallbacks :", fallback_specs(DEEP_MODEL) or "none (optional: MASTERY_FALLBACK_MODELS in .env)")
if not HAVE_DEEP_KEY:
    print(f"\n⚠️ No key for {DEEP_MODEL.split(':')[0]}: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")
if not HAVE_LANGSMITH_KEY:
    print("⚠️ No LANGSMITH_API_KEY: 🌐 LangSmith cells will print ⏭️ Skipped; offline twins run instead.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

In [ ]:
# 🧪 Helpers. Beta features emit LangChainBetaWarning: we let them print ONCE so you see them.
import warnings

from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver

from mastery.deep import build_deep_agent
from mastery.testing import ScriptedChatModel

warnings.simplefilter("once")


def calls(*items) -> AIMessage:
    return AIMessage("", tool_calls=[{"name": n, "args": a, "id": i} for n, a, i in items])

In [ ]:
# 🧪 Your installed versions vs the minimums the briefings quote (re-run after upgrading).
from importlib.metadata import PackageNotFoundError, version


def installed(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not installed"


table([
    {"feature": "F1 forked subagents", "needs": "deepagents>=0.7.13", "you have": installed("deepagents")},
    {"feature": "F4 RubricMiddleware", "needs": "deepagents>=0.6.5", "you have": installed("deepagents")},
    {"feature": "F3 interpreters", "needs": "deepagents[quickjs] (langchain-quickjs)", "you have": installed("langchain-quickjs")},
    {"feature": "F5 ACP", "needs": "deepagents-acp", "you have": installed("deepagents-acp")},
], ["feature", "needs", "you have"])

---
## F1 · Forked subagents (after 11.05)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** beta, requires `deepagents>=0.7.13`.

**What it is.** A subagent with `mode="fork"` inherits the parent's conversation and system prompt instead of receiving only a task description. The delegating call is replaced by a short continuation preamble; the answer comes back as a normal tool result.

**Use it when.** The subagent continues work the parent already did (drafting comments for issues already found, a postmortem for an investigation). Keep `isolated` (the default) for independent research: it's the context-isolation lever from Lesson 11.06.

> ⚠️ **Production note.** A fork pays for the whole parent history on every call, which is the opposite of isolation. Watch tokens on free tiers.

In [ ]:
# 🧪 F1 · isolated vs fork: what does the subagent actually see?
def run(mode):
    writer_model = ScriptedChatModel(responses=[AIMessage("Drafted two review comments.")])
    coordinator = ScriptedChatModel(responses=[
        AIMessage("Found: tokens logged in plaintext; no expiry check on refresh."),
        calls(("task", {"description": "Draft review comments", "subagent_type": "writer"}, "t")),
        AIMessage("Comments drafted.")])
    writer = {"name": "writer", "description": "Drafts PR review comments.", "model": writer_model, "mode": mode}
    if mode == "isolated":
        writer["system_prompt"] = "You draft PR review comments."
    agent = build_deep_agent(coordinator, subagents=[writer], checkpointer=InMemorySaver(), fallbacks=False)
    config = {"configurable": {"thread_id": mode}}
    agent.invoke({"messages": [{"role": "user", "content": "Review PR #482."}]}, config)
    agent.invoke({"messages": [{"role": "user", "content": "Now draft the comments."}]}, config)
    return writer_model.calls[0]


for mode in ("isolated", "fork"):
    seen = run(mode)
    print(f"{mode:<8} sees {len(seen)} messages:")
    for m in seen:
        print(f"   {m.type:<6} {m.text[:70]!r}")

**What just happened?** The isolated writer saw 2 messages (its own prompt and the task). The forked writer saw the whole conversation, including the parent's findings, plus a continuation preamble. Its system message is the **parent's** (empty here, because this scripted coordinator has none), not a prompt of its own.

---
## F2 · Async subagents (after 11.05 and 11.10)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** beta; needs an Agent Protocol server (a LangSmith Deployment, or `langgraph dev` locally). Not runnable offline in this notebook.

**What it is.** `AsyncSubAgent` specs point at graphs on an Agent Protocol server. The supervisor gets five tools (`start_async_task`, `check_async_task`, `update_async_task`, `cancel_async_task`, `list_async_tasks`), launches work in the background and keeps talking to the user.

**Use it when.** Long tasks (minutes to hours) the user shouldn't wait for, or that you want to steer while they run.

> ⚠️ **Production note.** Background work needs durable state, worker capacity (one slot per running subagent with `langgraph dev`) and a UI that shows progress. Without those, a synchronous `task` is simpler and easier to evaluate.

---
## F3 · Interpreters and dynamic subagents (after 11.09)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** beta; `pip install -U "deepagents[quickjs]"`. Not installed by the course.

**What it is.** Interpreter middleware adds an `eval` tool: the agent writes JavaScript that runs in a QuickJS context. With programmatic tool calling (PTC) that code can call allowed tools; with **dynamic subagents** it can call `task()` in a loop to fan work out across many items and merge the results.

**Use it when.** Many similar sub-tasks (review every file, triage a batch of tickets) where a loop in code beats dozens of model turns.

> ⚠️ **Production note.** Code the model writes is untrusted: keep the PTC allowlist minimal, cap loop sizes and subagent counts, and evaluate fan-out runs for cost as well as quality.

---
## F4 · Grading rubrics (after 11.11)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** beta, requires `deepagents>=0.6.5`.

**What it is.** `RubricMiddleware(model=grader)` runs a grader sub-agent whenever the agent would finish. You pass a `rubric` in the input; the grader returns `satisfied`, `needs_revision` (its feedback goes back to the agent) or `failed`, up to `max_iterations`.

**Use it when.** Deliverables with checkable criteria (sections present, every claim cited, tests pass), where one more iteration is cheaper than a human review.

> ⚠️ **Production note.** Self-grading is not evaluation: a grader can be fooled the same way the writer was. Validate the grader against human labels (Lesson 10.04), cap iterations, and count grader tokens in your budget.

In [ ]:
# 🧪 F4 · RubricMiddleware: a grader checks the answer against a rubric and sends it back until satisfied.
from deepagents import RubricMiddleware


def verdict(result, criteria, explanation):
    return AIMessage("", tool_calls=[{"name": "GraderResponse", "id": result,
                                      "args": {"result": result, "explanation": explanation, "criteria": criteria}}])


grader = ScriptedChatModel(responses=[
    verdict("needs_revision", [{"name": "three lines", "passed": True},
                               {"name": "theme is spring", "passed": False, "gap": "it is about winter"}],
            "The poem is about winter, not spring."),
    verdict("satisfied", [{"name": "three lines", "passed": True}, {"name": "theme is spring", "passed": True}], "All criteria pass.")])
writer = ScriptedChatModel(responses=[AIMessage("cold wind\nempty field\nwinter sleeps"),
                                      AIMessage("soft rain\nspring blossoms open\nwarm light")])
grades = []
agent = build_deep_agent(writer, checkpointer=InMemorySaver(), fallbacks=False,
                         middleware=[RubricMiddleware(model=grader, max_iterations=3, on_evaluation=grades.append)])
out = agent.invoke({"messages": [{"role": "user", "content": "Write a haiku about spring."}],
                    "rubric": "- three lines\n- the theme is spring"}, {"configurable": {"thread_id": "rubric"}})
for g in grades:
    print(f"iteration {g['iteration']}: {g['result']:<15} {g['explanation']}")
print("final answer:", out["messages"][-1].text.replace(chr(10), " / "))
print("model calls (writer):", len(writer.calls), "| grader calls:", len(grader.calls))

**What just happened?** The first draft failed "theme is spring"; the grader's explanation went back to the writer as a message, the second draft passed, and the loop stopped. Two writer calls and two grader calls: a rubric at least doubles the calls of a run.

---
## F5 · Agent protocols: AG-UI, ACP and A2A (after 11.10)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** AG-UI and ACP have Deep Agents integration pages (ACP: `pip install deepagents-acp`); A2A is an endpoint of LangSmith Deployments. Check each page's install line and status before use.

**What it is.** **AG-UI** turns a run into a typed event stream for any AG-UI frontend (agent ↔ user interface). **ACP** connects a deep agent to code editors and IDEs (agent ↔ editor). **A2A** lets other agents call yours (agent ↔ agent). **MCP** (Module 05) is different again: agent ↔ tools.

**Use it when.** You need an existing client: an AG-UI frontend, an ACP-capable editor, or another team's agent. Otherwise the SSE mapping from Lesson 11.10 is simpler.

> ⚠️ **Production note.** A protocol endpoint is a public API: authentication, rate limits and the same HITL rules apply as for your own UI.

---
## F6 · Managed Deep Agents and a Deep Agents Code tour (after 11.11)

> 🧪 **Status (2026-10-16, `deepagents` 0.7.26):** Managed Deep Agents is a **public beta** on LangSmith Cloud (the `mda` CLI); the docs now call it the recommended production path. Deep Agents Code (`dcode`) is not officially supported on Windows (WSL only).

**What it is.** **Managed Deep Agents** runs the same harness for you: you declare instructions, tools, skills and model as a project of files, and the runtime supplies checkpointer, store, memory and deployment. **Deep Agents Code** is LangChain's terminal coding agent built on the SDK: worth a tour to see skills, memory, sandboxes and HITL combined in a real product.

**Use it when.** Managed: you want threads, memory and deployment without running infrastructure, and a beta is acceptable. `dcode`: reading a production-grade deep agent (optional, in WSL on Windows).

> ⚠️ **Production note.** Beta hosted service: check data residency, pricing and SLA, and keep your evals (11.11) runnable outside it so you can move.

## ⚠️ Pitfalls

| Symptom | Cause | Fix |
|---|---|---|
| `LangChainBetaWarning` | You used a beta feature | Fine for learning; on a critical path, pin and add tests |
| Fork run much more expensive | Fork resends the whole parent history | Isolate unless continuation matters |
| Rubric loop never ends | Grader keeps asking for revisions | `max_iterations`; sharper rubric; validated grader |
| `start_async_task` hangs locally | Not enough `langgraph dev` workers | Increase the worker pool (one slot per subagent + 1) |

## 🏋️ Exercises

1. **(Easy)** In F1, add a `system_prompt` to the forked writer. What does the docs table say it becomes for a fork?
2. **(Medium)** In F4, change the grader so the second verdict is `failed`. What does the run return?
3. **(Medium)** For each of F2–F6, write one sentence: would you use it in Project 11? Why or why not?
4. **(Stretch)** Pick one briefing, read its page again, and write the status line as of today, with your `deepagents` version.

<details>
<summary><b>Show solutions</b></summary>

**1.** For `mode: "fork"`, `system_prompt` is optional and acts as a fork-only addendum; the fork keeps the parent's system prompt.

**2.** The loop stops at `failed`: the last answer is the second draft, and `grades` records the `failed` verdict, so your app must check the verdict, not just the final message.

**3.** A reasonable answer: F4 yes (a rubric for "every claim cited"), with a validated grader; F2 no (runs are minutes, not hours); F3 maybe (fan-out over many documents), with caps; F5 no (no external client); F6 optional for deployment.

**4.** Open the page, compare it with the status line, and log differences in `docs/research-log.md` with the date and versions.
</details>

## 🔎 Docs drill (5 min)

Which RubricMiddleware verdicts end the loop, and what happens on needs_revision?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [Grading rubrics](https://docs.langchain.com/oss/python/deepagents/rubric), or read `??deepagents.RubricMiddleware`.
2. `satisfied` and `failed` end it; `max_iterations` caps it.
3. On `needs_revision`, the grader's feedback is injected as a `HumanMessage` and the agent loop resumes.

</details>

## ✅ Checkpoint

- [ ] I can explain each of the six features in one sentence and say when I'd use it.
- [ ] I ran the fork and rubric demos and can explain their costs.
- [ ] I know where each status line comes from and how to re-check it.
- [ ] Restart & Run All passes without keys; outputs cleared before saving.

## 🔎 Research checkpoint

Open the [Deep Agents changelog](https://docs.langchain.com/oss/python/deepagents/changelog-py). Has any of F1–F6 left beta since 2026-10-16? Update the status lines in your notes and log the change with versions in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Frontier features solve real problems (continuation, background work, code fan-out, self-checks, protocols, hosting) but change fast.
- Date every claim, pin versions, and keep beta features behind your own wrapper and evals.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: the **module project**: read [project/README.md](project/README.md) (spec, modes, acceptance criteria), then open [`11-project-deep-research-analyst.ipynb`](project/11-project-deep-research-analyst.ipynb).

[Module 11 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)